# Migrate Containers from ECR to DockerHub

This notebook:
1. Lists all containers on ECR
2. Filters ECR images by push date (only includes images pushed after a specified date)
3. Lists all containers on DockerHub
4. Finds containers that are on ECR but not on DockerHub
5. Pulls those containers from ECR
6. Pushes them to DockerHub

## Setup and Imports

In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
import os
import base64
from datetime import datetime, timezone
import docker
import boto3
from botocore.exceptions import ClientError
from datasmith.docker.dockerhub import _list_dockerhub_tags_single_repo, _get_dockerhub_credentials
from datasmith.docker.ecr import _list_ecr_tags_single_repo
from datasmith.logging_config import configure_logging

logger = configure_logging()

# Configuration
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
ECR_REPO = "formulacode/all"
DOCKERHUB_NAMESPACE = "formulacode"  # Change to your DockerHub namespace
DOCKERHUB_REPO = "all"

# Date filter - only migrate images pushed after this date
MIN_PUSHED_AT_UTC = datetime(2025, 11, 7, 1, 0, 0, tzinfo=timezone.utc)

# Get DockerHub credentials
DOCKERHUB_USERNAME = os.environ.get("DOCKERHUB_USERNAME")
DOCKERHUB_PASSWORD = os.environ.get("DOCKERHUB_TOKEN") or os.environ.get("DOCKERHUB_PASSWORD")

# Get Docker client
docker_client = docker.from_env()

# Get AWS account ID for ECR
session = boto3.session.Session(region_name=AWS_REGION)
sts = session.client("sts")
account_id = sts.get_caller_identity()["Account"]
ECR_REGISTRY = f"{account_id}.dkr.ecr.{AWS_REGION}.amazonaws.com"

print(f"AWS Account ID: {account_id}")
print(f"ECR Registry: {ECR_REGISTRY}")
print(f"ECR Repository: {ECR_REPO}")
print(f"DockerHub Namespace: {DOCKERHUB_NAMESPACE}")
print(f"DockerHub Repository: {DOCKERHUB_REPO}")
print(f"Date Filter: Only images pushed after {MIN_PUSHED_AT_UTC}")

/mnt/sdd1/atharvas/formulacode/datasmith


18:49:04 INFO     botocore.credentials: Found credentials in shared credentials file: ~/.aws/credentials


AWS Account ID: 204464138089
ECR Registry: 204464138089.dkr.ecr.us-east-1.amazonaws.com
ECR Repository: formulacode/all
DockerHub Namespace: formulacode
DockerHub Repository: all
Date Filter: Only images pushed after 2025-11-07 01:00:00+00:00


## List All Containers on ECR

In [2]:
print("Listing all images on ECR...")
ecr_tags = _list_ecr_tags_single_repo(region=AWS_REGION, repo_name=ECR_REPO)

print(f"\nFound {len(ecr_tags)} images on ECR")
print(f"\nFirst 10 ECR images:")
for tag in sorted(ecr_tags)[:10]:
    print(f"  - {ECR_REGISTRY}/{ECR_REPO}:{tag}")

18:49:05 INFO     botocore.credentials: Found credentials in shared credentials file: ~/.aws/credentials


Listing all images on ECR...

Found 1746 images on ECR

First 10 ECR images:
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:activitysim-activitysim-2f9afa0ad632d9dd3d98730c631daebb9f765b1a--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:allencellmodeling-aicsimageio-738ce92dcc4440563b77557bcc1fdf6b0a82738f--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:allencellmodeling-aicsimageio-738ce92dcc4440563b77557bcc1fdf6b0a82738f--run
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:allencellmodeling-aicsimageio-c49a613dc54381d11237240ba36f0ef54603a7d6--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:arviz-devs-arviz-821c126ed9a41777518be256f075efd859b8e62b--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:arviz-devs-arviz-cfbfbeb4b274a4990803c179936567d524f5e694--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:arviz-devs-arviz-cfbfbeb4b274a4990

In [3]:
def filter_ecr_tags_by_push_date(
    tags: set[str],
    *,
    region: str,
    repo_name: str,
    cutoff: datetime,
) -> set[str]:
    """
    Filter ECR tags to only include those pushed after the cutoff date.
    
    Args:
        tags: Set of ECR image tags to filter
        region: AWS region
        repo_name: ECR repository name
        cutoff: Only include images pushed after this datetime (must be timezone-aware)
    
    Returns:
        Set of tags that were pushed after the cutoff date
    """
    if not tags:
        return set()
    
    session = boto3.session.Session(region_name=region)
    ecr_client = session.client("ecr")
    
    kept: set[str] = set()
    
    # Query ECR in chunks of 100 tags (API limit)
    CHUNK_SIZE = 100
    tag_list = sorted(tags)
    
    for i in range(0, len(tag_list), CHUNK_SIZE):
        chunk = tag_list[i : i + CHUNK_SIZE]
        image_ids = [{"imageTag": tag} for tag in chunk]
        
        try:
            resp = ecr_client.describe_images(
                repositoryName=repo_name,
                imageIds=image_ids
            )
        except ClientError as ce:
            code = ce.response.get("Error", {}).get("Code")
            if code in {"RepositoryNotFoundException", "ImageNotFoundException"}:
                # Skip this chunk if repo/images don't exist
                logger.warning(f"Repository or images not found for chunk {i}-{i+len(chunk)}")
                continue
            # Re-raise unexpected errors
            raise
        
        # Process image details
        for detail in resp.get("imageDetails", []):
            pushed_at = detail.get("imagePushedAt")  # boto3 returns timezone-aware datetime
            image_tags = detail.get("imageTags", [])
            
            if not pushed_at or not image_tags:
                continue
            
            # If this image was pushed after the cutoff, keep all its tags
            if pushed_at > cutoff:
                for tag in image_tags:
                    if tag in tags:  # Only add if it was in our original set
                        kept.add(tag)
    
    return kept


print(f"Filtering {len(ecr_tags)} ECR images by push date (after {MIN_PUSHED_AT_UTC})...")
filtered_ecr_tags = filter_ecr_tags_by_push_date(
    ecr_tags,
    region=AWS_REGION,
    repo_name=ECR_REPO,
    cutoff=MIN_PUSHED_AT_UTC
)

print(f"\nFiltered from {len(ecr_tags)} to {len(filtered_ecr_tags)} images")
print(f"Removed {len(ecr_tags) - len(filtered_ecr_tags)} images pushed before {MIN_PUSHED_AT_UTC}")
print(f"\nFirst 10 filtered ECR images:")
for tag in sorted(filtered_ecr_tags)[:10]:
    print(f"  - {ECR_REGISTRY}/{ECR_REPO}:{tag}")

# Update ecr_tags to use the filtered set
ecr_tags = filtered_ecr_tags

18:49:12 INFO     botocore.credentials: Found credentials in shared credentials file: ~/.aws/credentials


Filtering 1746 ECR images by push date (after 2025-11-07 01:00:00+00:00)...

Filtered from 1746 to 953 images
Removed 793 images pushed before 2025-11-07 01:00:00+00:00

First 10 filtered ECR images:
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:allencellmodeling-aicsimageio-738ce92dcc4440563b77557bcc1fdf6b0a82738f--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:allencellmodeling-aicsimageio-c49a613dc54381d11237240ba36f0ef54603a7d6--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:arviz-devs-arviz-821c126ed9a41777518be256f075efd859b8e62b--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:arviz-devs-arviz-cfbfbeb4b274a4990803c179936567d524f5e694--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:arviz-devs-arviz-e6b5e2bbdcd721cb621e7171964d84e9d48d592c--final
  - 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:bjodah-chempy-2d148283c48a84d9414765fbabc806042654b9c2--fin

## Filter ECR Images by Push Date

Filter ECR images to only include those pushed after the specified date.

## List All Containers on DockerHub

In [4]:
if not DOCKERHUB_USERNAME or not DOCKERHUB_PASSWORD:
    raise ValueError(
        "DockerHub credentials required. Set DOCKERHUB_USERNAME and DOCKERHUB_TOKEN environment variables.\n"
        "Generate tokens at: https://hub.docker.com/settings/security"
    )

print("Listing all images on DockerHub...")
dockerhub_tags = _list_dockerhub_tags_single_repo(
    namespace=DOCKERHUB_NAMESPACE,
    repo_name=DOCKERHUB_REPO,
    username=DOCKERHUB_USERNAME,
    password=DOCKERHUB_PASSWORD
)

print(f"\nFound {len(dockerhub_tags)} images on DockerHub")
print(f"\nFirst 10 DockerHub images:")
for tag in sorted(dockerhub_tags)[:10]:
    print(f"  - docker.io/{DOCKERHUB_NAMESPACE}/{DOCKERHUB_REPO}:{tag}")

Listing all images on DockerHub...

Found 793 images on DockerHub

First 10 DockerHub images:
  - docker.io/formulacode/all:allencellmodeling-aicsimageio-738ce92dcc4440563b77557bcc1fdf6b0a82738f--final
  - docker.io/formulacode/all:allencellmodeling-aicsimageio-c49a613dc54381d11237240ba36f0ef54603a7d6--final
  - docker.io/formulacode/all:arviz-devs-arviz-821c126ed9a41777518be256f075efd859b8e62b--final
  - docker.io/formulacode/all:arviz-devs-arviz-cfbfbeb4b274a4990803c179936567d524f5e694--final
  - docker.io/formulacode/all:arviz-devs-arviz-e6b5e2bbdcd721cb621e7171964d84e9d48d592c--final
  - docker.io/formulacode/all:bjodah-chempy-2d148283c48a84d9414765fbabc806042654b9c2--final
  - docker.io/formulacode/all:danielgtaylor-python-betterproto-bd7de203e16e949666b2844b3dec1eb7c4ed523c--final
  - docker.io/formulacode/all:dasdae-dascore-6f098be09791468f989715a5bcfb651a0a44db3e--final
  - docker.io/formulacode/all:dasdae-dascore-7139676bd5c82db6456c0939022a5b14275b0dfb--final
  - docker.io/fo

## Find Containers on ECR but Not on DockerHub

In [5]:
# Find images that exist on ECR but not on DockerHub
missing_tags = ecr_tags - dockerhub_tags

print(f"\nFound {len(missing_tags)} images on ECR that are NOT on DockerHub")
print(f"\nFirst 20 missing images:")
for tag in sorted(missing_tags)[:20]:
    print(f"  - {tag}")

# Store sorted list for migration
tags_to_migrate = sorted(missing_tags)
print(f"\nTotal images to migrate: {len(tags_to_migrate)}")


Found 161 images on ECR that are NOT on DockerHub

First 20 missing images:
  - deepchecks-deepchecks-9d1261c40a3685ece57f0fefc72a36ed129fc10e--final
  - devitocodes-devito-77ffae75d3498c2d576d8aa42452d03d30c10535--final
  - devitocodes-devito-c7c5277ced30d737475b278610b8471a309bb8df--final
  - dipy-dipy-03976f506cca9bc0b38fa4c5ea264d780f000c24--final
  - dipy-dipy-0f32296de973299b21352370e1288633778d0949--final
  - dipy-dipy-b585dd6a502c27617804fdf6d73929573e7059f9--final
  - kedro-org-kedro-38bb1b248a49e9687eb60831508d30c018eea565--final
  - kedro-org-kedro-4ba6b50352454bf73fa6c1a62f9ae5a60b595aff--final
  - kedro-org-kedro-70734ce00ee46b58c85b4cf04afbe89d32c06758--final
  - lmfit-lmfit-py-03daaf8ac13158920745bceece36f28b5d599af9--final
  - lmfit-lmfit-py-03f1b9b9660ebd00177f6cb17d12121cdcccd1a5--final
  - lmfit-lmfit-py-1a009ed4fee493ed4fb5dedc7691fa01507de90e--final
  - lmfit-lmfit-py-589c029fff571397ea14d9dbe6ae2283c6f10780--final
  - microsoft-qcodes-6e3c05df1839d7a49dff967c45f3

## Authenticate with ECR and DockerHub

In [6]:
# Login to ECR
print("Logging in to ECR...")
ecr_client = session.client("ecr")
auth_response = ecr_client.get_authorization_token()
auth_data = auth_response["authorizationData"][0]
ecr_username, ecr_password = base64.b64decode(auth_data["authorizationToken"]).decode().split(":", 1)
ecr_endpoint = auth_data["proxyEndpoint"].replace("https://", "")

docker_client.login(
    username=ecr_username,
    password=ecr_password,
    registry=ecr_endpoint
)
print(f"✓ Logged in to ECR: {ecr_endpoint}")

# Login to DockerHub
print("\nLogging in to DockerHub...")
docker_client.login(
    username=DOCKERHUB_USERNAME,
    password=DOCKERHUB_PASSWORD,
    registry="docker.io"
)
print(f"✓ Logged in to DockerHub as {DOCKERHUB_USERNAME}")

Logging in to ECR...
✓ Logged in to ECR: 204464138089.dkr.ecr.us-east-1.amazonaws.com

Logging in to DockerHub...
✓ Logged in to DockerHub as formulacode


## Pull Containers from ECR

This cell pulls all missing containers from ECR. This may take a long time depending on the number and size of images.

In [ ]:
from tqdm.notebook import tqdm

pulled_images = {}
pull_errors = {}

print(f"Pulling {len(tags_to_migrate)} images from ECR...\n")

for tag in tqdm(tags_to_migrate, desc="Pulling from ECR"):
    ecr_image_ref = f"{ECR_REGISTRY}/{ECR_REPO}:{tag}"
    try:
        print(f"Pulling {ecr_image_ref}...")
        image = docker_client.images.pull(ecr_image_ref)
        pulled_images[tag] = image
        print(f"  ✓ Pulled {ecr_image_ref}")
    except Exception as e:
        pull_errors[tag] = str(e)
        print(f"  ✖ Failed to pull {ecr_image_ref}: {e}")

print(f"\n✓ Successfully pulled {len(pulled_images)} images")
if pull_errors:
    print(f"✖ Failed to pull {len(pull_errors)} images")
    print("\nPull errors:")
    for tag, error in list(pull_errors.items())[:10]:
        print(f"  - {tag}: {error}")

## Push Containers to DockerHub

This cell pushes all pulled containers to DockerHub. This may take a long time.

**Note:** Rate limiting may occur with DockerHub. The cell includes retry logic.

In [ ]:
print("=" * 80)
print("MIGRATION SUMMARY")
print("=" * 80)
print(f"Date Filter:                        Images pushed after {MIN_PUSHED_AT_UTC}")
print(f"Total images on ECR (filtered):     {len(ecr_tags)}")
print(f"Total images on DockerHub (before): {len(dockerhub_tags)}")
print(f"Images needing migration:           {len(tags_to_migrate)}")
print(f"Images successfully pulled:         {len(pulled_images)}")
print(f"Images successfully pushed:         {len(pushed_images)}")
print(f"Images failed to pull:              {len(pull_errors)}")
print(f"Images failed to push:              {len(push_errors)}")
print("=" * 80)

if pushed_images:
    print("\nSuccessfully migrated images:")
    for tag in sorted(pushed_images.keys())[:20]:
        print(f"  ✓ {tag}")
    if len(pushed_images) > 20:
        print(f"  ... and {len(pushed_images) - 20} more")

if pull_errors or push_errors:
    print("\n⚠ Some images failed to migrate. Review the errors above.")

## Summary

In [ ]:
print("=" * 80)
print("MIGRATION SUMMARY")
print("=" * 80)
print(f"Total images on ECR:               {len(ecr_tags)}")
print(f"Total images on DockerHub (before): {len(dockerhub_tags)}")
print(f"Images needing migration:          {len(tags_to_migrate)}")
print(f"Images successfully pulled:        {len(pulled_images)}")
print(f"Images successfully pushed:        {len(pushed_images)}")
print(f"Images failed to pull:             {len(pull_errors)}")
print(f"Images failed to push:             {len(push_errors)}")
print("=" * 80)

if pushed_images:
    print("\nSuccessfully migrated images:")
    for tag in sorted(pushed_images.keys())[:20]:
        print(f"  ✓ {tag}")
    if len(pushed_images) > 20:
        print(f"  ... and {len(pushed_images) - 20} more")

if pull_errors or push_errors:
    print("\n⚠ Some images failed to migrate. Review the errors above.")

## Optional: Clean Up Local Images

Uncomment and run this cell to remove the pulled images from local Docker storage to free up space.

In [ ]:
# print("Cleaning up local images...")
# for tag, image in pulled_images.items():
#     try:
#         docker_client.images.remove(image.id, force=True)
#         print(f"  Removed {tag}")
#     except Exception as e:
#         print(f"  Failed to remove {tag}: {e}")
# print("✓ Cleanup complete")